# DwellSense / NestIQ: Smart Hostel Recommendation Model
### Architecture: Gradient Boosted Decision Trees (XGBoost Regressor)
**Objective:** Predict a comprehensive 0–100 Smart Recommendation Score based on 7 operational & quality features.

The 7 core features evaluated:
1. `average_rating` (1.0 to 5.0 stars from verified, unflagged student reviews)
2. `complaint_resolution_rate` (0.0 to 1.0 proportion of resolved issues)
3. `review_volume` (number of verified reviews)
4. `num_bookings` (historical booking popularity)
5. `proximity` (distance in km from student's university campus)
6. `complaint_freq` (complaints relative to bookings)
7. `resolution_speed_hrs` (average hours taken to resolve student issues)

In [ ]:
# 1. Install & Import Libraries
import numpy as np
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('XGBoost and analytical packages loaded successfully!')

In [ ]:
# 2. Load or Generate the 7-Feature Hostel Training Dataset (1,000 Records)
dataset_filename = 'hostel_xgboost_dataset_1000.csv'

if os.path.exists(dataset_filename):
    print(f'Loading existing dataset: {dataset_filename}')
    df = pd.read_csv(dataset_filename)
else:
    print('Generating 1,000 realistic hostel records across quality tiers...')
    import random
    np.random.seed(42)
    random.seed(42)
    records = []
    for _ in range(1000):
        tier = random.choices(['high', 'mid', 'low', 'new'], weights=[0.25, 0.50, 0.20, 0.05])[0]
        if tier == 'high':
            rating = round(random.uniform(4.2, 4.95), 2)
            res_rate = round(random.uniform(0.82, 0.99), 2)
            reviews = random.randint(45, 280)
            bookings = random.randint(reviews, reviews + random.randint(30, 200))
            dist = round(random.uniform(0.3, 4.5), 2)
            complaint_freq = round(random.uniform(0.01, 0.10), 2)
            res_speed = round(random.uniform(6.0, 36.0), 1)
        elif tier == 'mid':
            rating = round(random.uniform(3.2, 4.19), 2)
            res_rate = round(random.uniform(0.55, 0.81), 2)
            reviews = random.randint(15, 120)
            bookings = random.randint(reviews, reviews + random.randint(10, 90))
            dist = round(random.uniform(1.2, 8.5), 2)
            complaint_freq = round(random.uniform(0.08, 0.28), 2)
            res_speed = round(random.uniform(24.0, 72.0), 1)
        elif tier == 'low':
            rating = round(random.uniform(1.8, 3.19), 2)
            res_rate = round(random.uniform(0.15, 0.54), 2)
            reviews = random.randint(8, 75)
            bookings = random.randint(reviews, reviews + random.randint(5, 40))
            dist = round(random.uniform(3.5, 16.0), 2)
            complaint_freq = round(random.uniform(0.25, 0.65), 2)
            res_speed = round(random.uniform(48.0, 140.0), 1)
        else:
            rating = round(random.uniform(3.8, 5.0), 2)
            res_rate = round(random.uniform(0.70, 1.0), 2)
            reviews = random.randint(1, 10)
            bookings = random.randint(1, 15)
            dist = round(random.uniform(0.5, 6.0), 2)
            complaint_freq = round(random.uniform(0.0, 0.12), 2)
            res_speed = round(random.uniform(12.0, 48.0), 1)

        c_rating = (rating / 5.0) * 30.0
        c_res = res_rate * 20.0
        c_rev = min(10.0, (np.log1p(reviews) / np.log1p(250)) * 10.0)
        c_book = min(10.0, (np.log1p(bookings) / np.log1p(450)) * 10.0)
        c_dist = 10.0 if dist <= 1.5 else (7.5 if dist <= 3.5 else (4.0 if dist <= 7.0 else (1.0 if dist <= 12.0 else -2.0)))
        c_freq = (1.0 - min(1.0, complaint_freq)) * 10.0
        c_speed = 10.0 if res_speed <= 24.0 else (7.0 if res_speed <= 48.0 else (4.0 if res_speed <= 72.0 else 1.0))
        noise = np.random.normal(0, 1.2)
        target = round(float(np.clip(c_rating + c_res + c_rev + c_book + c_dist + c_freq + c_speed + noise, 10.0, 99.5)), 1)
        records.append({
            'average_rating': rating, 'complaint_resolution_rate': res_rate,
            'review_volume': reviews, 'num_bookings': bookings, 'proximity': dist,
            'complaint_freq': complaint_freq, 'resolution_speed_hrs': res_speed,
            'target_score': target
        })
    df = pd.DataFrame(records)
    df.to_csv(dataset_filename, index=False)

print(f'Dataset Shape: {df.shape}')
display(df.head())

In [ ]:
# 3. Exploratory Data Analysis & Feature Correlation
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Feature Correlation Heatmap with Target Smart Score')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Train the XGBoost Regressor Model
feature_cols = [
    'average_rating', 'complaint_resolution_rate', 'review_volume',
    'num_bookings', 'proximity', 'complaint_freq', 'resolution_speed_hrs'
]
X = df[feature_cols].values
y = df['target_score'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.10,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate on unseen test set
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('='*50)
print('XGBOOST RECOMMENDATION MODEL METRICS')
print('='*50)
print(f'R² Score (Goodness of Fit) : {r2 * 100:.2f}%')
print(f'Mean Absolute Error (MAE)  : {mae:.2f} points (out of 100)')
print(f'Root Mean Squared Error    : {rmse:.2f}')
print('='*50)

In [ ]:
# 5. Visualizations: Feature Importance & Prediction Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature Importance
importances = model.feature_importances_
indices = np.argsort(importances)
axes[0].barh([feature_cols[i] for i in indices], importances[indices], color='teal')
axes[0].set_title('XGBoost Relative Feature Importance')
axes[0].set_xlabel('Relative Weight')

# Actual vs. Predicted Scatter Plot
axes[1].scatter(y_test, y_pred, alpha=0.6, color='indigo')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Smart Score')
axes[1].set_ylabel('Predicted Smart Score')
axes[1].set_title(f'Actual vs. Predicted Scores (R² = {r2*100:.1f}%)')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Export Decision Trees to Pure JSON (for Flutter & Render API)
config = json.loads(model.get_booster().save_config())
base_score = float(config['learner']['learner_model_param']['base_score'])
trees = [json.loads(t) for t in model.get_booster().get_dump(dump_format='json')]

json_model = {
    'base_score': base_score,
    'features': feature_cols,
    'trees': trees
}

with open('xgboost_model.json', 'w') as f:
    json.dump(json_model, f, indent=2)

print(f'Successfully exported xgboost_model.json with base_score={base_score:.2f} and {len(trees)} trees!')
# from google.colab import files
# files.download('xgboost_model.json')

In [ ]:
# 7. Interactive Recommendation Score Tester
def predict_smart_score(rating, res_rate, reviews, bookings, proximity_km, complaint_freq, speed_hrs):
    input_features = np.array([[rating, res_rate, reviews, bookings, proximity_km, complaint_freq, speed_hrs]])
    predicted_score = float(model.predict(input_features)[0])
    clamped = max(0.0, min(100.0, round(predicted_score, 1)))
    
    print('='*50)
    print('HOSTEL INPUT PROFILE:')
    print(f'  - Verified Rating         : {rating} / 5.0')
    print(f'  - Complaint Resolution    : {res_rate * 100:.0f}%')
    print(f'  - Review Volume           : {reviews} reviews')
    print(f'  - Total Bookings          : {bookings} bookings')
    print(f'  - Distance from Campus    : {proximity_km} km')
    print(f'  - Complaint Frequency     : {complaint_freq * 100:.1f}%')
    print(f'  - Resolution Speed        : {speed_hrs} hours')
    print('-'*50)
    print(f'🌟 PREDICTED SMART SCORE   : {clamped} / 100.0')
    print('='*50)

# Try 3 different hostel tiers
print('1. Premier Hostel near Campus:')
predict_smart_score(rating=4.8, res_rate=0.95, reviews=120, bookings=180, proximity_km=1.2, complaint_freq=0.03, speed_hrs=12.0)

print('\n2. Average Hostel further away:')
predict_smart_score(rating=3.5, res_rate=0.65, reviews=35, bookings=50, proximity_km=6.5, complaint_freq=0.20, speed_hrs=48.0)

print('\n3. Low-rated Hostel with slow complaint resolution:')
predict_smart_score(rating=2.1, res_rate=0.25, reviews=15, bookings=20, proximity_km=14.0, complaint_freq=0.55, speed_hrs=110.0)